In [1]:
!pip install -q -U --force-reinstall pyarrow
!pip install -q -U pymupdf datasets "transformers==4.45.2" accelerate peft bitsandbytes sentencepiece torchao

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
# Packages have been successfully installed above.

In [3]:
# 2. Imports
import os
import re
import gc
import json
import unicodedata
from dataclasses import dataclass,asdict
from typing import List,Dict,Any
import fitz
import torch
from datasets import Dataset,DatasetDict
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from peft import(
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

W1122 17:23:00.485000 6705 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:52: FutureWarning: Support for Transformers < 5.6 is deprecated in PEFT and will be removed in the next release, v1.0. You have Transformers 4.45.2. Please upgrade to Transformers >= 5.6.
  warnings.warn(


In [4]:
@dataclass
class Config:
  pdf_path:str="/content/Metformin-Lipid-Therapy-Knowledge.pdf"
  model_name:str= "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
  #output directories
  output_dir:str="/content/phrama_t"
  adapter_dir:str="/content/pharma_tinyllama_lora_adapter"
  processed_data_dir:str="/content/pharma_processed_data"
  #Text preprocessing
  min_char_per_paragraph: int=80
  block_size: int=512
  #train/eval split
  test_size:float=0.15
  seed:int=42
  #Lora parameter
  lora_r:int=16
  lora_alph:int=32
  lora_dropout:float=0.05

  #Training parameters
  num_train_epochs:float=3.0
  per_device_train_batch_size:int=1
  pre_device_eval_batch_size:int=1
  gradient_accumulation_steps:int=8
  learning_rate:float=2e-4
  warmup_ratio:float=0.03
  weight_decay:float=0.01
  logging_steps:int=5
  eval_steps:int=10
  save_steps:int=25
  save_total_limit:int=2
  max_steps:int=-1


In [5]:
config=Config()

In [6]:
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/phrama_t', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_char_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alph=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, pre_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, logging_steps=5, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [7]:
os.makedirs(config.output_dir,exist_ok=True)
os.makedirs(config.adapter_dir,exist_ok=True)
os.makedirs(config.processed_data_dir,exist_ok=True)

In [8]:
# optional colab upload helper
if not os.path.exists(config.pdf_path):
  print(f"Pdf not found at: {config.pdf_path}")
else:
  print(f"Pdf found at: {config.pdf_path}")

Pdf found at: /content/Metformin-Lipid-Therapy-Knowledge.pdf


In [9]:
# Extract text from PDF
def extract_pdf_pages(pdf_path:str)->List[Dict[str,Any]]:
  pages=[]
  with fitz.open(pdf_path)as doc:
    for page_index,page in enumerate(doc,start=1):
      text=page.get_text("text")
      text=text.strip()if text else ""
      if text:
        pages.append({"page_number":page_index,"text":text,"char_count":len(text),})
  return pages


In [10]:
pdf_pages=extract_pdf_pages(config.pdf_path)

In [11]:
for item in pdf_pages:
  print(f"Page {item['page_number']}: {item['char_count']} characters")

Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [12]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

In [13]:
# Text Cleaning utilities

In [14]:
def normalize_unicode(text:str)->str:
  return unicodedata.normalize("NFKC",text)

In [15]:
def clean_pdf_text(text:str)->str:
  text=normalize_unicode(text)
  text=text.replace("\u200b","")
  text=text.replace("\ufeff","")
  text=re.sub(r"(\w)-\n(\w)",r"\1\2",text)
  text = re.sub(r"[ \t]+", " ", text)

  text = re.sub(r"\n{3,}", "\n\n", text)

  text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
  paragraphs = re.split(r"\n\s*\n", text)

  cleaned_paragraphs = []

  for paragraph in paragraphs:
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            cleaned_paragraphs.append(paragraph)

  return "\n\n".join(cleaned_paragraphs)

In [16]:
pdf_pages

[{'page_number': 1,
  'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin

In [17]:
cleanded_page=[]

In [18]:
for page in pdf_pages:
  cleanned_text=clean_pdf_text(page['text'])
  cleanded_page.append({"page_number":page["page_number"],"text":cleanned_text,"char_count":len(cleanned_text),})

In [19]:
cleanned_text

'Pharma Domain Training Data - Page 6 Page 6 - Clinical Trials, Pharmacovigilance, and Regulatory Data Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Clinical trial phases Clinical development usually progresses through multiple phases. Phase I trials often focus on safety, tolerability, pharmacokinetics, and dose escalation in a small number of participants. Phase II trials explore preliminary efficacy, dose selection, and continued safety. Phase III trials evaluate efficacy and safety in larger populations, often comparing the investigational product with standard of care or placebo. Post-marketing studies and real-world evidence can further characterize long-term safety and effectiveness. Trial design language Important clinical trial concepts include randomization, blinding, control arm, inclusion criteria, exclusion criteria, primary endpoint, secondary endpoint, adverse event, serious adverse event, sta

In [20]:
cleanded_page

[{'page_number': 1,
  'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.\n\nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing

In [21]:
# Split cleaned pages into paragraphs
def split_into_paragraphs_records(cleaned_pages: List[Dict[str, Any]], min_char: int = 80) -> List[Dict[str, Any]]:
    records = []
    seen=set()
    for page in cleaned_pages:
      paragraphs=re.split(r"\n\s*\n",page["text"])
      for paragraph_index,paragraph in enumerate(paragraphs,start=1):
        if len(paragraph)<min_char:
          continue
        normalized_key=re.sub(r"\s+"," ",paragraph.strip().lower())
        if normalized_key in seen:
          continue
        seen.add(normalized_key)
        records.append({
            "text":paragraph,
            "source_page":[page["page_number"]],
            "paragraph_id":paragraph_index,
            "char_count":len(paragraph),
        })

    return records

In [22]:
paragraph_records=split_into_paragraphs_records(cleanded_page)

In [23]:
print(f"Total paragraph records: {len(paragraph_records)}")
for i,record in enumerate(paragraph_records):
  print("="*100)
  print(f"Paragraph {i+1}: {record['char_count']} characters")
  print(record["text"])


Total paragraph records: 9
Paragraph 1: 575 characters
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Paragraph 2: 598 characters
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastati

In [24]:
# Save the extracted and cleaned corpus for auditability
raw_page_path=os.path.join(config.processed_data_dir,"raw_pages_raw.jsonl")
prag_record_path=os.path.join(config.processed_data_dir,"paragraph_records.jsonl")
with open(raw_page_path,"w")as f:
  for page in pdf_pages:
    f.write(json.dumps(page,ensure_ascii=False)+"\n")#
with open(prag_record_path,"w")as f:
  for record in paragraph_records:
    f.write(json.dumps(record,ensure_ascii=False)+"\n")

In [25]:
# Creating huggning face datasets
if len(paragraph_records)<2:
  raise ValueError("Not enough paragraph records to create a dataset")
text_dataset=Dataset.from_list(paragraph_records)

In [26]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [27]:
tokanizer=AutoTokenizer.from_pretrained(config.model_name,use_fast=True)#


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [28]:
if tokanizer.pad_token is None:
  tokanizer.pad_token_id = tokanizer.eos_token_id
tokanizer.padding_side = "right"
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokanizer)}")
print(f"Pad token: {tokanizer.pad_token} | Pad token id: {tokanizer.pad_token_id}")
print(f"EOS token: {tokanizer.eos_token} | EOS token id: {tokanizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


In [29]:
def tokenize_function(examples):
  return tokanizer(examples["text"],truncation=True)

In [30]:
#
def group_texts(examples):
  concatenated_examples={k:sum(examples[k],[])for k in examples.keys()}
  total_length=len(concatenated_examples["input_ids"])
  total_length=(total_length//config.block_size)*config.block_size
  if total_length==0:
    return{k:[]for k in concatenated_examples.keys()}
  result={
      k:t[i:i+config.block_size]for i in range(0,total_length,config.block_size)
      for k,t in concatenated_examples.items()
  }
  result["labels"]=result["input_ids"].copy()
  return result

In [31]:
# Create train/test split from text_dataset
raw_datasets = text_dataset.train_test_split(
    test_size=config.test_size,
    seed=config.seed
)

# Tokenize raw datasets
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

In [32]:
#
final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing tokenized text into blocks of {config.block_size}",
)

Packing tokenized text into blocks of 512:   0%|          | 0/7 [00:00<?, ? examples/s]

Packing tokenized text into blocks of 512:   0%|          | 0/2 [00:00<?, ? examples/s]

In [33]:
if len(final_datasets["train"])==0:
  raise ValueError(
      "No training blocks were created, Reduce config.block_size or provide a larger corpus"

  )

In [34]:
sample=final_datasets['train'][0]

In [35]:
sample

{'input_ids': 4414, 'attention_mask': 1, 'labels': 4414}

In [36]:
# Load base model
#
use_cuda=torch.cuda.is_available()
print("CUDA availabe:",use_cuda)
if use_cuda:
  print("Device name:",torch.cuda.get_device_name(0))

CUDA availabe: True
Device name: Tesla T4


In [37]:
# Clear memory before loading the model
#
gc.collect()
if use_cuda:
  torch.cuda.empty_cache()
if use_cuda:
  bnb_config=BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_use_double_quant=True,
  )
  model=AutoModelForCausalLM.from_pretrained(
      config.model_name,
      quantization_config=bnb_config,
      device_map="auto",
  )
  base_model=prepare_model_for_kbit_training(model)
else:
  base_model=AutoModelForCausalLM.from_pretrained(
      config.model_name,
      torch_dtype=torch.float32,
      trust_remote_code=True,

  )
base_model.config.use_cache=False
print("Base model loaded")

Base model loaded


In [38]:
# Apply LoRA adapters
#
lora_config=LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alph,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
)

In [39]:
model=get_peft_model(base_model,lora_config)


In [40]:
model.print_trainable_parameters()

trainable params: 13,160,448 || all params: 1,113,208,832 || trainable%: 1.1822


In [41]:
# Data collator
data_collator=DataCollatorForLanguageModeling(tokenizer=tokanizer,mlm=False)

In [42]:
# Training arguments
import inspect
from transformers import TrainingArguments
eval_key = "test" if "test" in final_datasets else "validation"
eval_model = "steps" if len(final_datasets[eval_key]) > 0 else "no"
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.pre_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,
    fp16=True if use_cuda else False,
    bf16=False if use_cuda else True,
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps if len(final_datasets[eval_key]) > 0 else None,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    report_to="none",
    remove_unused_columns=False,
    run_name="pharma_tinyllama_lora",
)
training_args_params = inspect.signature(TrainingArguments).parameters
if "eval_strategy" in training_args_params:
  training_kwargs["eval_strategy"] = eval_model
elif "eval_steps" in training_args_params:
  training_kwargs["evaluation_strategy"] = eval_model
safe_training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in training_args_params
}

In [43]:
remove_args=set(training_kwargs.keys())-set(training_args_params.keys())
if remove_args:
  print(f"Removed args: {removed_args}")



In [44]:
training_args=TrainingArguments(**safe_training_kwargs)

In [45]:
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
evaluation_str

In [46]:
#Build trainer
import inspect
from transformers import Trainer
trainer_kwargs=dict(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets[eval_key],
    data_collator=data_collator,


)
trainer_params=inspect.signature(Trainer).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokanizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokanizer

safe_trainer_kwargs={
    key:value
    for key,value in trainer_kwargs.items()
    if key in trainer_params
}
removed_trainer_args=set(trainer_kwargs.keys())-set(trainer_params.keys())
if removed_trainer_args:
  print(f"Removed args: {removed_trainer_args}")

In [47]:
trainer=Trainer(**safe_trainer_kwargs)
print("Trainer is ready")

Trainer is ready


In [48]:
# Re-pack the dataset properly to ensure inputs are nested sequence lists
def group_texts_fixed(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])
    if total_length >= config.block_size:
        total_length = (total_length // config.block_size) * config.block_size
    else:
        # Ensure we have at least one block by keeping it short as a sequence list
        total_length = total_length

    result = {
        k: [t[i : i + config.block_size] for i in range(0, total_length, config.block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Map with the corrected packing function
final_datasets = tokenized_datasets.map(
    group_texts_fixed,
    batched=True,
    desc="Correctly packing tokenized text into blocks",
)

# Re-build trainer with the fixed datasets and unpatched model
safe_trainer_kwargs["model"] = model
safe_trainer_kwargs["train_dataset"] = final_datasets["train"]
safe_trainer_kwargs["eval_dataset"] = final_datasets[eval_key]
trainer = Trainer(**safe_trainer_kwargs)

# Start training
train_result = trainer.train()
print("Training completed")
print(train_result)

Correctly packing tokenized text into blocks:   0%|          | 0/7 [00:00<?, ? examples/s]

Correctly packing tokenized text into blocks:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:242: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss


/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:347: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Training completed
TrainOutput(global_step=3, training_loss=1.074186642964681, metrics={'train_runtime': 18.6118, 'train_samples_per_second': 0.967, 'train_steps_per_second': 0.161, 'total_flos': 38621411278848.0, 'train_loss': 1.074186642964681, 'epoch': 2.0})


In [49]:
# Save adapter and tokenizer
trainer.model.save_pretrained(config.adapter_dir)
tokanizer.save_pretrained(config.adapter_dir)
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:347: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['tokenizer.json', 'tokenizer_config.json', 'tokenizer.model', 'adapter_config.json', 'adapter_model.safetensors', 'README.md', 'special_tokens_map.json']


In [51]:
from huggingface_hub import notebook_login

# 1. Authenticate with your Hugging Face Write Token
print("Please provide a Hugging Face token with WRITE permissions:")
notebook_login()

# 2. push LoRA adapter to Hugging Face hub
repo_id="piyushg1122/pharma-tinyllama-domain-lora"
model.push_to_hub(
    repo_id,
    private=True
)
tokanizer.push_to_hub(repo_id,private=True)
print(f"LoRA adapter pushed to: {repo_id}")

Please provide a Hugging Face token with WRITE permissions:


README.md: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:347: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|9         | 29.8MB /  315MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p9l_t7fwr/tokenizer.model: 100%|##########|  500kB /  500kB            

LoRA adapter pushed to: piyushg1122/pharma-tinyllama-domain-lora


In [52]:
# Clean old object to free memory
del trainer
try:
  del model
  del base_model
except NameError:
  pass
gc.collect()
if use_cuda:
  torch.cuda.empty_cache()


In [53]:
if use_cuda:
  reload_bnb_config=BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_use_double_quant=True,
  )
  interface_base_model=AutoModelForCausalLM.from_pretrained(
      config.model_name,
      quantization_config=reload_bnb_config,
      device_map="auto",
  )
else:
  interface_base_model=AutoModelForCausalLM.from_pretrained(
      config.model_name,
      torch_dtype=torch.float32,
      trust_remote_code=True,
  )

# Ensure the tokenizer is loaded
interface_tokenizer=AutoTokenizer.from_pretrained(config.model_name,use_fast=True)
if interface_tokenizer.pad_token is None:
  interface_tokenizer.pad_token_id=interface_tokenizer.eos_token_id

# Correctly load the fine-tuned adapter on top of the base model for both CPU and GPU paths
inference_model=PeftModel.from_pretrained(
    interface_base_model,
    config.adapter_dir,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None,
)
inference_model.eval()
print("Model loaded")

Model loaded


In [54]:
# Inference helper
def generate_completion(prompt:str,max_new_tokens:int=120)->str:
  device="cuda"if torch.cuda.is_available()else"cpu"
  inputs=interface_tokenizer(prompt,return_tensors="pt")
  inputs={k:v.to(device)for k,v in inputs.items()}
  with torch.no_grad():
    outputs=inference_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=interface_tokenizer.eos_token_id,
        eos_token_id=interface_tokenizer.eos_token_id,
    )
  return interface_tokenizer.decode(outputs[0],skip_special_tokens=True)

In [55]:
# Test text continuation

# Ensure interface_tokenizer is loaded regardless of the conditional path in previous cells
try:
    interface_tokenizer
except NameError:
    print("Initializing interface_tokenizer...")
    interface_tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
    if interface_tokenizer.pad_token is None:
        interface_tokenizer.pad_token_id = interface_tokenizer.eos_token_id

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is used to treat type 2 diabetes mellitus (T2DM) and other conditions such as non-insulin dependent diabetes mellitus (NIDDM).
Metformin is a member of the class of drugs known as sulfonylureas, which are also used for the treatment of T2DM. The drug works by increasing the production of insulin in the body. This helps to control blood sugar levels.
The drug is available in tablet form.
What is Metformin?

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe is more effective than either drug alone.
"The combination of Atorvastatin and Ezetimibe has been shown to be more effective in reducing LDL cholesterol levels, which is the primary risk factor for cardiovascular disease," said Dr. David A. Katz, director of the Yale-Griffin Prevention Research Center and profes

In [56]:
# Optional merge stop
merged_model_dir = "/content/pharma_tinyllama_merged_model"
merged_model = inference_model.merge_and_unload()
merged_model.save_pretrained(merged_model_dir)
interface_tokenizer.save_pretrained(merged_model_dir)
print(f"Merged model saved to: {merged_model_dir}")

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:387: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Merged model saved to: /content/pharma_tinyllama_merged_model


In [ ]:
import shutil
from google.colab import files

# 1. Compress the merged model folder
print("Zipping merged model...")
shutil.make_archive("/content/pharma_tinyllama_merged_model", 'zip', "/content/pharma_tinyllama_merged_model")

# 2. Compress the lora adapter folder
print("Zipping LoRA adapters...")
shutil.make_archive("/content/pharma_tinyllama_lora_adapter", 'zip', "/content/pharma_tinyllama_lora_adapter")

# 3. Compress the processed data folder
print("Zipping processed data...")
shutil.make_archive("/content/pharma_processed_data", 'zip', "/content/pharma_processed_data")

# 4. Trigger downloads in the browser
print("Triggering browser downloads...")
files.download("/content/pharma_tinyllama_merged_model.zip")
files.download("/content/pharma_tinyllama_lora_adapter.zip")
files.download("/content/pharma_processed_data.zip")
print("Download requests sent successfully!")

In [58]:
import os, json, gc, math, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# free GPU memory left over from earlier cells
for name in ["model", "base_model", "interface_base_model", "inference_model", "merged_model", "trainer"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

# same split as training (seed 42)
records = [json.loads(l) for l in open(os.path.join(config.processed_data_dir, "paragraph_records.jsonl"))]
ds = Dataset.from_list(records).train_test_split(test_size=config.test_size, seed=config.seed)
train_texts, test_texts = ds["train"]["text"], ds["test"]["text"]

tok = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

@torch.no_grad()
def score(model, texts):
    """Token-weighted perplexity. Each paragraph is its own sequence (no packing)."""
    model.eval()
    total_nll, total_tokens, per_par = 0.0, 0, []
    for t in texts:
        ids = tok(t, return_tensors="pt", truncation=True, max_length=2048)["input_ids"].to("cuda")
        n = ids.shape[1] - 1                      # first token is not predicted
        loss = model(input_ids=ids, labels=ids).loss.item()
        total_nll += loss * n
        total_tokens += n
        per_par.append(math.exp(loss))
    return math.exp(total_nll / total_tokens), total_tokens, per_par

base = AutoModelForCausalLM.from_pretrained(config.model_name, quantization_config=bnb, device_map="auto")
base_test, n_test, base_test_pp = score(base, test_texts)
base_train, n_train, _ = score(base, train_texts)

ft = PeftModel.from_pretrained(base, config.adapter_dir)
ft_test, _, ft_test_pp = score(ft, test_texts)
ft_train, _, _ = score(ft, train_texts)

def row(name, tokens, b, f):
    return f"{name:<28}{tokens:>8}{b:>12.2f}{f:>14.2f}{(f-b)/b*100:>+10.1f}%"

print(f"{'Set':<28}{'Tokens':>8}{'Base PPL':>12}{'Fine-tuned':>14}{'Change':>11}")
print(row(f"Held-out ({len(test_texts)} paragraphs)", n_test, base_test, ft_test))
print(row(f"Train ({len(train_texts)} paragraphs)", n_train, base_train, ft_train))
print("\nPer-paragraph held-out PPL (base -> fine-tuned):")
for b, f in zip(base_test_pp, ft_test_pp):
    print(f"  {b:.2f} -> {f:.2f}")

Set                           Tokens    Base PPL    Fine-tuned     Change
Held-out (2 paragraphs)          256        5.46          5.46      -0.0%
Train (7 paragraphs)            3412        7.85          7.85      -0.0%

Per-paragraph held-out PPL (base -> fine-tuned):
  9.13 -> 9.13
  3.71 -> 3.71


In [59]:
print(max(p.abs().max().item() for n, p in ft.named_parameters() if "lora_B" in n))


0.00012030253856210038
